# Military Service and Immigrants' Integration: Evidence from the Vietnam Draft Lotteries

**Authors:** Nan Zhang, Melissa M. Lee

**Journal:** American Political Science Review (2026)

This notebook reproduces the analysis from the paper above.
It was auto-generated by [PoliRep](https://polirep.org).

---

## Setup

Install packages and import standard libraries.

In [ ]:
!pip install -q pandas numpy matplotlib scipy statsmodels pyfixest

import pandas as pd
import numpy as np
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

## Configuration & Constants

Paper-specific constants used by the analysis code below.

In [ ]:
# Paper-specific constants (extracted from config.py)

# Colab-friendly data path (replaces local PAPER_ROOT-based path)
DATA_CONVERTED = Path("data/converted")
OUTPUT_DIR = Path("output")
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"

BIRTH_COHORTS = [1949, 1950, 1951, 1952]  # Main analysis cohorts
ALL_COHORTS = [1948, 1949, 1950, 1951, 1952, 1953]  # Extended range
SAMPLES = {
    "pooled": "sample_main1",  # All origins, 1949-1952
    "western": "sample_main2",  # Western origin, 1949-1952
    "nonwestern": "sample_main3",  # Non-Western origin, 1949-1952
}
PRIMARY_OUTCOMES = [
    "naturalized",
    "res_integrate_tract",
    "res_integrate_blkgrp",
    "only_engl",
    "engl_ability",
    "spouse_notconatl",
    "spouse_native",
]
SECONDARY_OUTCOMES = [
    "married",
    "spouse_whitenative",
    "college_some",
    "college_grad",
    "income_total",
    "unemployed",
]
TREATMENT_VAR = "veteran"
INSTRUMENT_VAR = "draft_risk"
PANEL_VAR = "bpl"  # Birthplace (country of origin)
CONTROLS = ["age_immig"]  # Plus birth year/month dummies
SIMULATION = {
    "n_obs_pooled": 28500,
    "n_obs_western": 9700,
    "n_obs_nonwestern": 18800,
    "veteran_rate": 0.16,
    "compliance_rate": 0.07,
    "n_birthplaces": 120,
}
TOLERANCE = {
    "strict": 1e-6,  # For exact matches
    "loose": 1e-3,  # For coefficient comparisons
    "relaxed": 0.05,  # For summary statistics
}

## Data Preparation


### Load Main Data

Load or generate the main analysis dataset. For this restricted-access
paper, attempts to load actual Census 2000 SEDF data if available
(for users with FSRDC access), otherwise generates simulated data.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load or generate the main analysis dataset.

    For restricted-access papers, this function attempts to load real data
    if available (e.g., for users with FSRDC access), otherwise generates
    simulated data for demonstration.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Check if actual data exists
    analysis_file = DATA_CONVERTED / "analysis.parquet"

    if analysis_file.exists():
        warnings.warn("Loading actual restricted data - ensure FSRDC approval")
        return pd.read_parquet(analysis_file)
    else:
        warnings.warn(
            "Restricted data not available - generating simulated data. "
            "Results are for demonstration only and do not reflect actual findings."
        )
        return _generate_simulated_data()

In [ ]:
df = load_main_data()
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns[:20])}...")
print(f"Birth year range: {int(df['birth_year'].min())}-{int(df['birth_year'].max())}")
print(f"Veteran rate: {df['veteran'].mean():.1%}")
print(f"Draft risk rate: {df['draft_risk'].mean():.1%}")
display(df.head())


### Create Draft Risk Instrument

Merge Vietnam draft lottery data (RSN and APN by birth date) and
create binary draft_risk indicator (1 if RSN ≤ APN). This is the
instrumental variable for veteran status.

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load or generate the main analysis dataset.

    For restricted-access papers, this function attempts to load real data
    if available (e.g., for users with FSRDC access), otherwise generates
    simulated data for demonstration.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Check if actual data exists
    analysis_file = DATA_CONVERTED / "analysis.parquet"

    if analysis_file.exists():
        warnings.warn("Loading actual restricted data - ensure FSRDC approval")
        return pd.read_parquet(analysis_file)
    else:
        warnings.warn(
            "Restricted data not available - generating simulated data. "
            "Results are for demonstration only and do not reflect actual findings."
        )
        return _generate_simulated_data()

In [ ]:
df = load_main_data()
print(f"RSN range: {int(df['rsn'].min())}-{int(df['rsn'].max())}")
print(f"APN by cohort:")
print(df.groupby('birth_year')['apn'].first().to_dict())
print(f"Draft risk rate: {df['draft_risk'].mean():.1%}")
print(f"Compliance rate: {(df[df['draft_risk']==1]['veteran'].mean() - df[df['draft_risk']==0]['veteran'].mean()):.1%}")
display(df[['birth_year', 'birth_month', 'birth_day', 'rsn', 'apn', 'draft_risk']].head(10))


### Create Integration Outcomes

Construct integration outcome variables: naturalization (binary),
residential integration at tract and block group levels (continuous),
English language outcomes (binary and ordinal), and spouse
characteristics (conditional on marriage).

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load or generate the main analysis dataset.

    For restricted-access papers, this function attempts to load real data
    if available (e.g., for users with FSRDC access), otherwise generates
    simulated data for demonstration.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Check if actual data exists
    analysis_file = DATA_CONVERTED / "analysis.parquet"

    if analysis_file.exists():
        warnings.warn("Loading actual restricted data - ensure FSRDC approval")
        return pd.read_parquet(analysis_file)
    else:
        warnings.warn(
            "Restricted data not available - generating simulated data. "
            "Results are for demonstration only and do not reflect actual findings."
        )
        return _generate_simulated_data()

In [ ]:
df = load_main_data()
print("Integration outcome means:")
print(f"  Naturalized: {df['naturalized'].mean():.1%}")
print(f"  Res integrate (tract): {df['res_integrate_tract'].mean():.3f}")
print(f"  Res integrate (blkgrp): {df['res_integrate_blkgrp'].mean():.3f}")
print(f"  Only English: {df['only_engl'].mean():.1%}")
print(f"  English ability (mean): {df['engl_ability'].mean():.2f}")
print(f"  Non-conatl spouse (married): {df['spouse_notconatl'].mean():.1%}")
print(f"  Native spouse (married): {df['spouse_native'].mean():.1%}")
display(df[['naturalized', 'res_integrate_tract', 'only_engl', 'spouse_native']].describe())


### Create Socioeconomic Outcomes

Construct secondary outcomes: marriage (binary), education
(some college, college grad), total income (in $10,000s),
and unemployment (binary).

In [ ]:
def load_main_data() -> pd.DataFrame:
    """Load or generate the main analysis dataset.

    For restricted-access papers, this function attempts to load real data
    if available (e.g., for users with FSRDC access), otherwise generates
    simulated data for demonstration.

    Returns:
        DataFrame with all variables needed for analysis.
    """
    # Check if actual data exists
    analysis_file = DATA_CONVERTED / "analysis.parquet"

    if analysis_file.exists():
        warnings.warn("Loading actual restricted data - ensure FSRDC approval")
        return pd.read_parquet(analysis_file)
    else:
        warnings.warn(
            "Restricted data not available - generating simulated data. "
            "Results are for demonstration only and do not reflect actual findings."
        )
        return _generate_simulated_data()

In [ ]:
df = load_main_data()
print("Socioeconomic outcome means:")
print(f"  Married: {df['married'].mean():.1%}")
print(f"  Some college: {df['college_some'].mean():.1%}")
print(f"  College grad: {df['college_grad'].mean():.1%}")
print(f"  Income total (mean): ${df['income_total'].mean():.1f}k")
print(f"  Unemployed: {df['unemployed'].mean():.1%}")
display(df[['married', 'college_some', 'college_grad', 'income_total']].describe())


### Apply Sample Restrictions

Filter to analysis sample: men born abroad 1949-1952, immigrated
before draft year. Create sample indicators for pooled, Western,
and non-Western subsamples. Create birth year and month dummies
for fixed effects.

In [ ]:
def prepare_analysis_sample(df: pd.DataFrame, sample: str = "pooled") -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().
        sample: Which sample to use - "pooled", "western", or "nonwestern"

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    try:
        from .config import SAMPLES
    except ImportError:
        from config import SAMPLES

    # Select appropriate sample
    sample_var = SAMPLES[sample]
    analysis = df[df[sample_var] == True].copy()  # noqa: E712

    # Create birth year and month dummies for FE
    for year in BIRTH_COHORTS:
        analysis[f"byear_{year}"] = (analysis["birth_year"] == year).astype(int)

    for month in range(1, 13):
        analysis[f"bmonth_{month}"] = (analysis["birth_month"] == month).astype(int)

    return analysis

def get_sample_stats(sample: pd.DataFrame) -> dict:
    """Compute summary statistics for the analysis sample.

    Args:
        sample: Prepared analysis sample from prepare_analysis_sample().

    Returns:
        Dictionary with keys like n_obs, n_units, year_range, etc.
    """
    stats = {
        "n_obs": len(sample),
        "n_birthplaces": sample[PANEL_VAR].nunique(),
        "year_range": f"{sample['birth_year'].min()}-{sample['birth_year'].max()}",
        "veteran_rate": sample["veteran"].mean(),
        "draft_risk_rate": sample["draft_risk"].mean(),
        "compliance_rate": (
            sample[sample["draft_risk"] == 1]["veteran"].mean()
            - sample[sample["draft_risk"] == 0]["veteran"].mean()
        ),
    }
    return stats

In [ ]:
sample = prepare_analysis_sample(load_main_data(), sample="pooled")
print(f"Pooled sample shape: {sample.shape}")
stats = get_sample_stats(sample)
print(f"Observations: {stats['n_obs']}")
print(f"Birthplaces: {stats['n_birthplaces']}")
print(f"Year range: {stats['year_range']}")
print(f"Veteran rate: {stats['veteran_rate']:.1%}")
print(f"Draft risk rate: {stats['draft_risk_rate']:.1%}")
print(f"Compliance rate: {stats['compliance_rate']:.1%}")
display(sample.head())


### Subsample: Western Origin

Create Western-origin subsample (Europe, Canada, Australia, NZ, Israel).
This subsample has ~9,700 observations and is used for heterogeneity
analysis in Table 3.

In [ ]:
def prepare_analysis_sample(df: pd.DataFrame, sample: str = "pooled") -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().
        sample: Which sample to use - "pooled", "western", or "nonwestern"

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    try:
        from .config import SAMPLES
    except ImportError:
        from config import SAMPLES

    # Select appropriate sample
    sample_var = SAMPLES[sample]
    analysis = df[df[sample_var] == True].copy()  # noqa: E712

    # Create birth year and month dummies for FE
    for year in BIRTH_COHORTS:
        analysis[f"byear_{year}"] = (analysis["birth_year"] == year).astype(int)

    for month in range(1, 13):
        analysis[f"bmonth_{month}"] = (analysis["birth_month"] == month).astype(int)

    return analysis

def get_sample_stats(sample: pd.DataFrame) -> dict:
    """Compute summary statistics for the analysis sample.

    Args:
        sample: Prepared analysis sample from prepare_analysis_sample().

    Returns:
        Dictionary with keys like n_obs, n_units, year_range, etc.
    """
    stats = {
        "n_obs": len(sample),
        "n_birthplaces": sample[PANEL_VAR].nunique(),
        "year_range": f"{sample['birth_year'].min()}-{sample['birth_year'].max()}",
        "veteran_rate": sample["veteran"].mean(),
        "draft_risk_rate": sample["draft_risk"].mean(),
        "compliance_rate": (
            sample[sample["draft_risk"] == 1]["veteran"].mean()
            - sample[sample["draft_risk"] == 0]["veteran"].mean()
        ),
    }
    return stats

In [ ]:
sample_west = prepare_analysis_sample(load_main_data(), sample="western")
print(f"Western sample shape: {sample_west.shape}")
stats = get_sample_stats(sample_west)
print(f"Observations: {stats['n_obs']}")
print(f"Veteran rate: {stats['veteran_rate']:.1%}")
print(f"Compliance rate: {stats['compliance_rate']:.1%}")
print(f"Top birthplaces:")
print(sample_west['bpl'].value_counts().head())


### Subsample: Non-Western Origin

Create non-Western subsample (Mexico, Latin America, Asia, Africa,
US territories). This subsample has ~15,000-18,500 observations and
is used for heterogeneity analysis in Table 4.

In [ ]:
def prepare_analysis_sample(df: pd.DataFrame, sample: str = "pooled") -> pd.DataFrame:
    """Apply sample restrictions and construct derived variables.

    Args:
        df: Raw loaded DataFrame from load_main_data().
        sample: Which sample to use - "pooled", "western", or "nonwestern"

    Returns:
        Analysis-ready DataFrame with all filters and transformations applied.
    """
    try:
        from .config import SAMPLES
    except ImportError:
        from config import SAMPLES

    # Select appropriate sample
    sample_var = SAMPLES[sample]
    analysis = df[df[sample_var] == True].copy()  # noqa: E712

    # Create birth year and month dummies for FE
    for year in BIRTH_COHORTS:
        analysis[f"byear_{year}"] = (analysis["birth_year"] == year).astype(int)

    for month in range(1, 13):
        analysis[f"bmonth_{month}"] = (analysis["birth_month"] == month).astype(int)

    return analysis

def get_sample_stats(sample: pd.DataFrame) -> dict:
    """Compute summary statistics for the analysis sample.

    Args:
        sample: Prepared analysis sample from prepare_analysis_sample().

    Returns:
        Dictionary with keys like n_obs, n_units, year_range, etc.
    """
    stats = {
        "n_obs": len(sample),
        "n_birthplaces": sample[PANEL_VAR].nunique(),
        "year_range": f"{sample['birth_year'].min()}-{sample['birth_year'].max()}",
        "veteran_rate": sample["veteran"].mean(),
        "draft_risk_rate": sample["draft_risk"].mean(),
        "compliance_rate": (
            sample[sample["draft_risk"] == 1]["veteran"].mean()
            - sample[sample["draft_risk"] == 0]["veteran"].mean()
        ),
    }
    return stats

In [ ]:
sample_nonwest = prepare_analysis_sample(load_main_data(), sample="nonwestern")
print(f"Non-Western sample shape: {sample_nonwest.shape}")
stats = get_sample_stats(sample_nonwest)
print(f"Observations: {stats['n_obs']}")
print(f"Veteran rate: {stats['veteran_rate']:.1%}")
print(f"Compliance rate: {stats['compliance_rate']:.1%}")
print(f"Top birthplaces:")
print(sample_nonwest['bpl'].value_counts().head())


## Descriptive Statistics


### Descriptive Statistics

Summary statistics for pooled, Western-origin, and Non-Western-origin immigrant samples (Table A4.1)

In [ ]:
def run(sample: pd.DataFrame = None, sample_name: str = "pooled") -> dict:
    """Compute descriptive statistics for the analysis sample.

    Args:
        sample: Analysis sample DataFrame (if None, will load from data module)
        sample_name: Sample identifier for output files

    Returns:
        Dictionary with summary statistics
    """
    if sample is None:
        from .data import load_main_data, prepare_analysis_sample

        df = load_main_data()
        sample = prepare_analysis_sample(df, sample=sample_name)

    print(f"\n{'='*60}")
    print(f"Descriptive Statistics: {sample_name.upper()}")
    print(f"{'='*60}\n")

    # Basic sample characteristics
    print(f"Total observations: {len(sample):,}")
    print(f"Birth years: {sample['birth_year'].min()}-{sample['birth_year'].max()}")
    print(f"Unique birthplaces: {sample['bpl'].nunique()}")
    print(f"Veteran rate: {sample[TREATMENT_VAR].mean():.3f}")
    print(f"Draft risk rate: {sample['draft_risk'].mean():.3f}")

    compliance = (
        sample[sample["draft_risk"] == 1][TREATMENT_VAR].mean()
        - sample[sample["draft_risk"] == 0][TREATMENT_VAR].mean()
    )
    print(f"Compliance rate: {compliance:.3f}\n")

    # Variables to summarize
    key_vars = [
        TREATMENT_VAR,
        "draft_risk",
        "age_immig",
        "yrs_since_immig",
    ] + PRIMARY_OUTCOMES + SECONDARY_OUTCOMES

    # Filter to variables that exist
    key_vars = [v for v in key_vars if v in sample.columns]

    # Compute summary statistics
    print("Computing summary statistics...")
    stats = compute_summary_stats(sample, key_vars, weights="pwt")

    print("\n" + "="*80)
    print(stats.to_string(index=False))
    print("="*80)

    # Save to CSV and LaTeX
    csv_file = f"descriptives_{sample_name}.csv"
    stats.to_csv(TABLE_DIR / csv_file, index=False)
    print(f"\nSaved: {TABLE_DIR / csv_file}")

    # Create LaTeX table
    latex = stats.to_latex(
        index=False,
        float_format="%.3f",
        caption=f"Descriptive Statistics: {sample_name.title()} Sample",
        label=f"tab:desc_{sample_name}",
    )
    tex_file = f"descriptives_{sample_name}.tex"
    save_table(latex, tex_file, TABLE_DIR)

    # Cross-tabulation: Veteran status by draft risk
    print("\n" + "="*60)
    print("CROSS-TABULATION: Veteran Status by Draft Risk")
    print("="*60)

    crosstab = pd.crosstab(
        sample["draft_risk"],
        sample[TREATMENT_VAR],
        normalize="index",
        margins=True,
    )
    print(crosstab)

    crosstab_counts = pd.crosstab(
        sample["draft_risk"],
        sample[TREATMENT_VAR],
        margins=True,
    )
    print("\nCounts:")
    print(crosstab_counts)

    # Save crosstab
    crosstab_file = f"crosstab_veteran_{sample_name}.csv"
    crosstab_counts.to_csv(TABLE_DIR / crosstab_file)
    print(f"\nSaved: {TABLE_DIR / crosstab_file}")

    return {
        "stats": stats,
        "crosstab": crosstab,
        "n_obs": len(sample),
        "compliance_rate": compliance,
    }

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## First-Stage Analysis


### First-Stage Analysis

Instrument relevance: draft lottery → veteran status. Estimates compliance rates and weak instrument tests (Table 1, Table A6.2)

In [ ]:
def run(sample: pd.DataFrame = None, sample_name: str = "pooled") -> dict:
    """Run first-stage analysis.

    Args:
        sample: Analysis sample DataFrame (if None, will load from data module)
        sample_name: Sample identifier for output files

    Returns:
        Dictionary with results
    """
    if sample is None:
        from .data import load_main_data, prepare_analysis_sample

        df = load_main_data()
        sample = prepare_analysis_sample(df, sample=sample_name)

    print(f"\n{'='*60}")
    print(f"First Stage Analysis: {sample_name.upper()}")
    print(f"{'='*60}\n")

    # Build control list (age_immig + birth year/month dummies)
    controls = CONTROLS.copy()
    for year in BIRTH_COHORTS:
        controls.append(f"byear_{year}")
    for month in range(1, 13):
        controls.append(f"bmonth_{month}")

    # Remove one category from each set of dummies to avoid collinearity
    controls.remove("byear_1949")
    controls.remove("bmonth_1")

    # Run first stage
    print(f"Dependent variable: {TREATMENT_VAR}")
    print(f"Instrument: {INSTRUMENT_VAR}")
    print("Controls: age_immig + birth year FE + birth month FE")
    print(f"Entity effects: {PANEL_VAR} (birthplace)")
    print(f"Sample size: {len(sample):,}\n")

    result = first_stage_diagnostics(
        data=sample,
        treatment=TREATMENT_VAR,
        instrument=INSTRUMENT_VAR,
        controls=controls,
        entity_effects=PANEL_VAR,
    )

    # Extract results
    model = result["model"]
    f_stat = result["f_statistic"]
    coef = result["coef"]

    print(f"First-stage coefficient on {INSTRUMENT_VAR}: {coef:.4f}")
    if hasattr(model, "std_errors"):
        se = model.std_errors.get(INSTRUMENT_VAR, None)
        if se:
            print(f"Standard error: {se:.4f}")
            print(f"t-statistic: {coef/se:.2f}")

    if f_stat:
        print(f"F-statistic: {f_stat:.2f}")

    # Compute compliance rate (reduced form)
    compliance = (
        sample[sample[INSTRUMENT_VAR] == 1][TREATMENT_VAR].mean()
        - sample[sample[INSTRUMENT_VAR] == 0][TREATMENT_VAR].mean()
    )
    print(f"Compliance rate (LATE denominator): {compliance:.4f}")

    # Save results to CSV
    results_df = pd.DataFrame(
        {
            "Sample": [sample_name],
            "N": [len(sample)],
            "Draft_Risk_Coef": [coef],
            "F_Statistic": [f_stat if f_stat else "N/A"],
            "Compliance_Rate": [compliance],
        }
    )

    output_file = f"first_stage_{sample_name}.csv"
    results_df.to_csv(TABLE_DIR / output_file, index=False)
    print(f"\nSaved: {TABLE_DIR / output_file}")

    return {
        "model": model,
        "coefficient": coef,
        "f_statistic": f_stat,
        "compliance_rate": compliance,
        "n_obs": len(sample),
    }

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Main IV Analysis


### Main IV Analysis

2SLS estimates of military service → integration outcomes. OLS, ITT, and IV specifications for 7 integration measures (Tables 2-4)

In [ ]:
def run(sample: pd.DataFrame = None, sample_name: str = "pooled") -> dict:
    """Run main IV analysis for all integration outcomes.

    Args:
        sample: Analysis sample DataFrame (if None, will load from data module)
        sample_name: Sample identifier for output files

    Returns:
        Dictionary with results for each outcome
    """
    if sample is None:
        from .data import load_main_data, prepare_analysis_sample

        df = load_main_data()
        sample = prepare_analysis_sample(df, sample=sample_name)

    print(f"\n{'='*60}")
    print(f"Main IV Analysis: {sample_name.upper()}")
    print(f"{'='*60}\n")

    # Build control list
    controls = CONTROLS.copy()
    for year in BIRTH_COHORTS:
        controls.append(f"byear_{year}")
    for month in range(1, 13):
        controls.append(f"bmonth_{month}")

    # Remove reference categories
    controls.remove("byear_1949")
    controls.remove("bmonth_1")

    # Store results
    results = {}

    # For each outcome, run OLS, ITT (reduced form), and 2SLS
    for outcome in PRIMARY_OUTCOMES:
        print(f"\nOutcome: {outcome}")
        print("-" * 40)

        # Check if outcome exists and has variation
        if outcome not in sample.columns:
            print(f"  WARNING: {outcome} not in data, skipping")
            continue

        outcome_data = sample[outcome].dropna()
        if len(outcome_data) < 100:
            print(f"  WARNING: Too few observations ({len(outcome_data)}), skipping")
            continue

        # (1) Naive OLS: outcome ~ veteran + controls + bpl FE
        print("  Running OLS...")
        try:
            ols_result = run_ols(sample, outcome, TREATMENT_VAR, controls, PANEL_VAR)
            ols_coef = ols_result.params.get(TREATMENT_VAR, None)
            ols_se = ols_result.std_errors.get(TREATMENT_VAR, None) if hasattr(
                ols_result, "std_errors"
            ) else None
            print(f"    OLS coef: {ols_coef:.4f} ({ols_se:.4f})")
        except Exception as e:
            print(f"    OLS failed: {e}")
            ols_result = None
            ols_coef = None
            ols_se = None

        # (2) ITT (reduced form): outcome ~ draft_risk + controls + bpl FE
        print("  Running ITT (reduced form)...")
        try:
            itt_result = run_ols(sample, outcome, INSTRUMENT_VAR, controls, PANEL_VAR)
            itt_coef = itt_result.params.get(INSTRUMENT_VAR, None)
            itt_se = itt_result.std_errors.get(INSTRUMENT_VAR, None) if hasattr(
                itt_result, "std_errors"
            ) else None
            print(f"    ITT coef: {itt_coef:.4f} ({itt_se:.4f})")
        except Exception as e:
            print(f"    ITT failed: {e}")
            itt_result = None
            itt_coef = None
            itt_se = None

        # (3) 2SLS: outcome ~ veteran (instrumented) + controls + bpl FE
        print("  Running 2SLS...")
        try:
            iv_result = run_2sls(
                sample, outcome, TREATMENT_VAR, INSTRUMENT_VAR, controls, PANEL_VAR
            )
            iv_coef = iv_result.params.get(TREATMENT_VAR, None)
            iv_se = iv_result.std_errors.get(TREATMENT_VAR, None)
            print(f"    2SLS coef: {iv_coef:.4f} ({iv_se:.4f})")
        except Exception as e:
            print(f"    2SLS failed: {e}")
            iv_result = None
            iv_coef = None
            iv_se = None

        # Store results
        results[outcome] = {
            "ols_model": ols_result,
            "ols_coef": ols_coef,
            "ols_se": ols_se,
            "itt_model": itt_result,
            "itt_coef": itt_coef,
            "itt_se": itt_se,
            "iv_model": iv_result,
            "iv_coef": iv_coef,
            "iv_se": iv_se,
        }

    # Create summary table
    print("\n" + "="*60)
    print("SUMMARY TABLE")
    print("="*60)

    summary_rows = []
    for outcome, res in results.items():
        row = {
            "Outcome": outcome,
            "OLS_Coef": res["ols_coef"],
            "OLS_SE": res["ols_se"],
            "ITT_Coef": res["itt_coef"],
            "ITT_SE": res["itt_se"],
            "IV_Coef": res["iv_coef"],
            "IV_SE": res["iv_se"],
        }
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    print(summary_df.to_string(index=False))

    # Save to CSV
    output_file = f"main_iv_{sample_name}.csv"
    summary_df.to_csv(TABLE_DIR / output_file, index=False)
    print(f"\nSaved: {TABLE_DIR / output_file}")

    return results

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))

## Robustness Checks


### Robustness Checks

Attrition analysis, cohort-specific estimates, and exclusion restriction tests (Tables A8.4-A11.7)

In [ ]:
def run(sample: pd.DataFrame = None, sample_name: str = "pooled") -> dict:
    """Run robustness checks.

    Args:
        sample: Analysis sample DataFrame (if None, will load from data module)
        sample_name: Sample identifier for output files

    Returns:
        Dictionary with robustness test results
    """
    if sample is None:
        from .data import load_main_data, prepare_analysis_sample

        df = load_main_data()
        sample = prepare_analysis_sample(df, sample=sample_name)

    print(f"\n{'='*60}")
    print(f"Robustness Checks: {sample_name.upper()}")
    print(f"{'='*60}\n")

    results = {}

    # -------------------------------------------------------------------------
    # 1. Excess mortality test (Table A8.4)
    # -------------------------------------------------------------------------
    print("\n" + "-"*60)
    print("1. EXCESS MORTALITY TEST")
    print("-"*60)
    print("Testing whether veterans have higher mortality than expected.")
    print("(In Census 2000, measuring survival to 2000 for Vietnam-era cohorts)\n")

    # For demonstration: assume we can calculate expected survival
    # In reality, this requires external mortality tables
    veteran_rate = sample[TREATMENT_VAR].mean()
    expected_rate = 0.16  # Hypothetical baseline from non-combat data

    n_obs = len(sample)
    mortality_test = proportion_test(veteran_rate, expected_rate, n_obs)

    print(f"Observed veteran rate: {mortality_test['observed']:.4f}")
    print(f"Expected rate (if no excess mortality): {mortality_test['expected']:.4f}")
    print(f"Z-statistic: {mortality_test['z_statistic']:.3f}")
    print(f"P-value: {mortality_test['p_value']:.4f}")

    if mortality_test['p_value'] > 0.05:
        print("=> No significant excess mortality detected")
    else:
        print("=> Significant difference detected (possible attrition bias)")

    results["mortality_test"] = mortality_test

    # -------------------------------------------------------------------------
    # 2. Attrition by nationality (Table A8.5)
    # -------------------------------------------------------------------------
    print("\n" + "-"*60)
    print("2. ATTRITION BY NATIONALITY")
    print("-"*60)
    print("Testing for differential attrition by birthplace.\n")

    # For each major birthplace, test if veteran rate differs from pooled
    top_birthplaces = sample["bpl"].value_counts().head(10).index

    attrition_tests = []
    for bpl_code in top_birthplaces:
        bpl_sample = sample[sample["bpl"] == bpl_code]
        n_bpl = len(bpl_sample)

        if n_bpl < 50:
            continue

        bpl_vet_rate = bpl_sample[TREATMENT_VAR].mean()
        test = proportion_test(bpl_vet_rate, veteran_rate, n_bpl)

        attrition_tests.append(
            {
                "Birthplace": int(bpl_code),
                "N": n_bpl,
                "Vet_Rate": bpl_vet_rate,
                "Z_Stat": test["z_statistic"],
                "P_Value": test["p_value"],
            }
        )

    attrition_df = pd.DataFrame(attrition_tests)
    print(attrition_df.to_string(index=False))

    # Save
    attrition_file = f"attrition_by_nationality_{sample_name}.csv"
    attrition_df.to_csv(TABLE_DIR / attrition_file, index=False)
    print(f"\nSaved: {TABLE_DIR / attrition_file}")

    results["attrition_tests"] = attrition_df

    # -------------------------------------------------------------------------
    # 3. Exclusion restriction: Subgroup analysis (Table A10.6)
    # -------------------------------------------------------------------------
    print("\n" + "-"*60)
    print("3. EXCLUSION RESTRICTION TEST")
    print("-"*60)
    print("Testing whether draft risk affects outcomes only through service.\n")
    print("Strategy: Estimate effect separately for veterans and non-veterans.")
    print("If exclusion holds, draft_risk should have no effect among non-veterans.\n")

    # Build controls
    controls = CONTROLS.copy()
    for year in BIRTH_COHORTS:
        controls.append(f"byear_{year}")
    for month in range(1, 13):
        controls.append(f"bmonth_{month}")
    controls.remove("byear_1949")
    controls.remove("bmonth_1")

    # Test on one outcome: naturalization
    test_outcome = "naturalized"

    if test_outcome in sample.columns:
        # Split by veteran status
        veterans = sample[sample[TREATMENT_VAR] == 1]
        non_veterans = sample[sample[TREATMENT_VAR] == 0]

        print(f"Testing outcome: {test_outcome}")
        print(f"Veterans: N={len(veterans):,}")
        print(f"Non-veterans: N={len(non_veterans):,}\n")

        exclusion_results = {}

        # For each subgroup, regress outcome on draft_risk (reduced form)
        from .helpers import run_ols

        for group_name, group_data in [("Veterans", veterans), ("Non-Veterans", non_veterans)]:
            if len(group_data) < 100:
                print(f"{group_name}: Too few observations, skipping")
                continue

            try:
                model = run_ols(group_data, test_outcome, INSTRUMENT_VAR, controls, PANEL_VAR)
                coef = model.params.get(INSTRUMENT_VAR, None)
                se = model.std_errors.get(INSTRUMENT_VAR, None) if hasattr(
                    model, "std_errors"
                ) else None

                print(f"{group_name}:")
                print(f"  Draft risk coef: {coef:.4f} (SE: {se:.4f})")
                print(f"  T-stat: {coef/se:.2f}" if se else "")

                exclusion_results[group_name] = {
                    "coef": coef,
                    "se": se,
                    "n_obs": len(group_data),
                }
            except Exception as e:
                print(f"{group_name}: Failed - {e}")

        # Interpretation
        if "Non-Veterans" in exclusion_results:
            nv_coef = exclusion_results["Non-Veterans"]["coef"]
            nv_se = exclusion_results["Non-Veterans"]["se"]
            if abs(nv_coef / nv_se) < 1.96:
                print("\n=> Exclusion restriction holds: No effect among non-veterans")
            else:
                print(
                    "\n=> WARNING: Draft risk affects outcome among non-veterans "
                    "(exclusion restriction violated)"
                )

        results["exclusion_test"] = exclusion_results

    return results

In [ ]:
result = run()
print(type(result))
if isinstance(result, dict):
    print(list(result.keys()))